# 🎯 Midway LoRA Training — Coder + Reasoner + Arbiter

Trains the three adapters for the Midway pipeline on Colab's GPU, then saves them to Google Drive.

| Adapter | Base model | Dataset | Epochs |
|---|---|---|---|
| **coder** | `Qwen2.5-Coder-7B-Instruct` | `combined_lora_dataset.jsonl` (9125) | 1 |
| **reasoner** | `Qwen2.5-Coder-7B-Instruct` | `reasoner_lora_dataset.jsonl` (3500) | 1 |
| **arbiter** | `DeepSeek-R1-Distill-Qwen-7B` | `arbiter_lora_dataset.jsonl` (9707) | 2 |

**Just hit Runtime → Run all.** Outputs (LoRA adapters + Ollama GGUF) land in `MyDrive/midway-lora/output/`.

> **Epochs:** coder + reasoner overfit (tail loss ≈ 0.02) → 1 epoch. Arbiter is underfit (loss ≈ 3.7) → keep 2.
> **Before a fresh run:** make sure the regenerated datasets (esp. `arbiter_lora_dataset.jsonl` @ 9707) are committed + pushed to GitHub — the notebook clones fresh.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE = '/content/drive/MyDrive/midway-lora'
OUT = os.path.join(DRIVE, 'output')
os.makedirs(OUT, exist_ok=True)
print('Output dir:', OUT)

In [ ]:
# Unsloth (QLoRA trainer) + the HF stack.
# If the plain `unsloth` wheel is missing, uncomment the git install below.
!pip install -q unsloth
!pip install -q transformers datasets accelerate peft trl bitsandbytes
# !pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
print('dependencies installed')

In [ ]:
# Clone the repo to get the trainer, config, and pre-generated datasets.
import os
if not os.path.isdir('/content/midway-pipeline'):
    !git clone --depth 1 https://github.com/FoggyGoofball/midway-pipeline.git
os.chdir('/content/midway-pipeline/lora generator')
print('cwd:', os.getcwd())

In [ ]:
# Sanity-check the datasets landed with the clone.
import os
for f in ['combined_lora_dataset.jsonl', 'reasoner_lora_dataset.jsonl', 'arbiter_lora_dataset.jsonl']:
    n = sum(1 for _ in open(f, encoding='utf-8')) if os.path.exists(f) else 0
    print(f'{f}: {n} samples')

## 1. Train the coder

Base: `Qwen2.5-Coder-7B-Instruct` · dataset: combined (SEARCH/REPLACE + contract + negative API + failure corpus + paging + signals).

In [ ]:
!python lora_fine_tune.py --dataset combined_lora_dataset.jsonl --output lora_output_coder --epochs 1 --train-on-inputs false --save-steps 100

## 2. Train the reasoner

Base: `Qwen2.5-Coder-7B-Instruct` · dataset: reasoner (signals + verdict). Tribunal debate belongs to the arbiter LoRA, not the Director.

In [ ]:
!python lora_fine_tune.py --dataset reasoner_lora_dataset.jsonl --output lora_output_reasoner --epochs 1 --train-on-inputs false --save-steps 100

## 3. Train the arbiter (supreme arbiter)

Base: `DeepSeek-R1-Distill-Qwen-7B` (a true reasoning model).

> ⚠️ If R1's chat template misbehaves during training, swap `--base-model` for `unsloth/Qwen2.5-Coder-7B-Instruct` and rerun just this cell.

In [ ]:
!python lora_fine_tune.py --dataset arbiter_lora_dataset.jsonl --output lora_output_arbiter --base-model unsloth/DeepSeek-R1-Distill-Qwen-7B --epochs 2 --train-on-inputs false --save-steps 100

## 4. Save everything to Drive

In [ ]:
import shutil, os, zipfile

OUT = '/content/drive/MyDrive/midway-lora/output'
os.makedirs(OUT, exist_ok=True)

# 1. Copy the LoRA adapters (small) AND the Ollama-ready GGUF dirs.
#    lora_fine_tune.py writes GGUFs to <output>_gguf/ — a SIBLING of <output>/.
for name in ['lora_output_coder', 'lora_output_reasoner', 'lora_output_arbiter']:
    src = os.path.abspath(name)
    if os.path.isdir(src):
        shutil.copytree(src, os.path.join(OUT, name), dirs_exist_ok=True,
                        ignore=shutil.ignore_patterns(
                            'model-*.safetensors', 'model.safetensors.index.json',
                            'checkpoint-*', '.cache', 'optimizer.pt', 'scheduler.pt',
                            'rng_state.pth', 'training_args.bin'))
        print('saved adapter', name)
    # the GGUF export lives in <name>_gguf/
    gguf = os.path.abspath(name + '_gguf')
    if os.path.isdir(gguf):
        shutil.copytree(gguf, os.path.join(OUT, name + '_gguf'), dirs_exist_ok=True)
        print('saved gguf', name + '_gguf')
    else:
        print('WARN: missing', name + '_gguf', '(trainer GGUF export did not run)')

# 2. Slim zip: adapters + GGUFs only (skip merged 16-bit weights + checkpoints).
zip_path = os.path.join(OUT, 'midway_lora_adapters.zip')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for name in ['lora_output_coder', 'lora_output_reasoner', 'lora_output_arbiter',
                 'lora_output_coder_gguf', 'lora_output_reasoner_gguf', 'lora_output_arbiter_gguf']:
        src = os.path.abspath(name)
        if not os.path.isdir(src):
            continue
        for root, _, files in os.walk(src):
            if 'checkpoint-' in root or '.cache' in root:
                continue
            for f in files:
                if f.startswith('model-') or f == 'model.safetensors.index.json':
                    continue
                full = os.path.join(root, f)
                z.write(full, os.path.relpath(full, os.path.dirname(src)))
print('zipped ->', zip_path)

## 5. Deploy to the Steam Deck (Ollama)

Each output dir contains `adapter_model.safetensors` + a merged GGUF (`unsloth.Q4_K_M.gguf`). Copy the GGUF to the Deck and:

```bash
ollama create midway-coder-lora -f Modelfile.coder
ollama create midway-reasoner-lora -f Modelfile.reasoner
ollama create midway-arbiter-lora -f Modelfile.arbiter
```

Then point the pipeline at them via `MIDWAY_CODER_MODEL` / `MIDWAY_REVIEWER_MODEL` / `MIDWAY_ARBITER_MODEL`.